In [ ]:
# Bibliotecas necessárias
import os
import pandas as pd
import pyodbc
import warnings
import matplotlib.pyplot as plt
import geopandas as gpd

# Ignora warnings
warnings.filterwarnings('ignore')

# Caminho para o arquivo com credenciais
usuario = os.getenv('USERNAME')
path = fr'C:\Users\{usuario}\OneDrive - CAIXA Consórcio\Documentos\SENHA_BANCO_DADOS'
os.chdir(path)

# Carrega credenciais
df_senhas = pd.read_excel('SENHAS.xlsx')
server, database, username, password = df_senhas.iloc[0, 0:4]

# Conexão com o banco de dados SQL Server
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password}'
)

# Consulta SQL
query_table = """
SELECT 
    FT.AnoMesRef,
    FT.ID_UF,
    FT.Tipo_Pessoa,
    DP.CD_InscricaoNacional,
    RD.NM_Bem,
    FT.ST_Adimplencia,
    FT.VL_Bem
FROM 
    FT0015_CarteiraCotas AS FT
LEFT JOIN 
    DM0013_Pessoas AS DP ON FT.ID_Pessoa = DP.ID_Pessoa
LEFT JOIN 
    DM0011_Bens AS RD ON FT.ID_Bem = RD.ID_Bem
WHERE 
    FT.AnoMesRef >= '202410'
"""

print("📥 Executando a consulta SQL...")
df = pd.read_sql(query_table, conn)
conn.close()

# Calcula inadimplência
df['Inadimplente'] = df['ST_Adimplencia'].apply(lambda x: 1 if x == 'Inadimplente' else 0)
inadimplencia_por_uf = df.groupby('ID_UF').agg(
    Total=('ST_Adimplencia', 'count'),
    Inadimplentes=('Inadimplente', 'sum')
).reset_index()

inadimplencia_por_uf['Perc_Inadimplencia'] = (
    inadimplencia_por_uf['Inadimplentes'] / inadimplencia_por_uf['Total']
) * 100

# Dicionário com nomes dos estados
uf_codigos = {
    11: 'Rondônia', 12: 'Acre', 13: 'Amazonas', 14: 'Roraima', 15: 'Pará', 16: 'Amapá', 17: 'Tocantins',
    21: 'Maranhão', 22: 'Piauí', 23: 'Ceará', 24: 'Rio Grande do Norte', 25: 'Paraíba', 26: 'Pernambuco',
    27: 'Alagoas', 28: 'Sergipe', 29: 'Bahia', 31: 'Minas Gerais', 32: 'Espírito Santo', 33: 'Rio de Janeiro',
    35: 'São Paulo', 41: 'Paraná', 42: 'Santa Catarina', 43: 'Rio Grande do Sul', 50: 'Mato Grosso do Sul',
    51: 'Mato Grosso', 52: 'Goiás', 53: 'Distrito Federal'
}
inadimplencia_por_uf['Estado'] = inadimplencia_por_uf['ID_UF'].map(uf_codigos)

# Carrega mapa dos estados do Brasil
estados = gpd.read_file('https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson')

# Une os dados de inadimplência com o mapa
mapa = estados.merge(inadimplencia_por_uf, left_on='name', right_on='Estado')

# Plot do mapa com heatmap
fig, ax = plt.subplots(figsize=(12, 10))
mapa.plot(
    column='Perc_Inadimplencia',
    cmap='RdYlGn_r',  # Vermelho para valores altos, verde para baixos
    linewidth=0.8,
    ax=ax,
    edgecolor='0.8',
    legend=True
)
ax.set_title('🗺️ Percentual de Inadimplência por Estado', fontsize=16)
ax.axis('off')
plt.show()


ModuleNotFoundError: No module named 'geopandas'

: 